# CSE422 Lab Project — Diabetes Prediction

**Dataset:** diabetes_dataset.csv

**Problem Type:** Binary Classification

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc, classification_report
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
print("Libraries loaded successfully.")

## Section 2: Load Dataset

In [ ]:
df = pd.read_csv('diabetes_dataset.csv')
print('Shape:', df.shape)
df.head()

## Section 3: Dataset Description

- **Number of Features:** 8 input features + 1 target (`diabetes`)
- **Problem Type:** Binary Classification — the target variable `diabetes` takes values 0 (no diabetes) or 1 (diabetes).
- **Number of Data Points:** 100,000
- **Quantitative Features:** `age`, `bmi`, `HbA1c_level`, `blood_glucose_level`, `hypertension`, `heart_disease`
- **Categorical Features:** `gender`, `smoking_history`
- **Encoding Needed:** Yes — `gender` and `smoking_history` are text-based categorical columns that must be encoded into numbers for ML models.

In [ ]:
print('Columns:', df.columns.tolist())
print()
print('Data Types:')
print(df.dtypes)
print()
print('Null Value Counts:')
print(df.isnull().sum())
print()
print('Basic Statistics:')
df.describe()

### 3.1 Correlation Heatmap (Seaborn)

In [ ]:
# Temporarily encode categoricals to compute correlation
df_corr = df.copy()
le_tmp = LabelEncoder()
df_corr['gender'] = le_tmp.fit_transform(df_corr['gender'].astype(str))
df_corr['smoking_history'] = le_tmp.fit_transform(df_corr['smoking_history'].astype(str))

plt.figure(figsize=(10, 7))
sns.heatmap(df_corr.corr(numeric_only=True), annot=True, fmt='.2f',
            cmap='coolwarm', square=True, linewidths=0.5)
plt.title('Feature Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.show()

print("\nCorrelation with target (diabetes):")
print(df_corr.corr(numeric_only=True)['diabetes'].sort_values(ascending=False))

**Observations from Correlation:**
- `blood_glucose_level` and `HbA1c_level` have the **strongest positive correlation** with diabetes (~0.42 and ~0.40), which is clinically expected.
- `age` and `bmi` also show moderate positive correlation.
- `gender` and `smoking_history` have very weak correlation with the target.

### 3.2 Class Distribution (Imbalanced Dataset)

In [ ]:
df_clean = df.dropna(subset=['diabetes']).copy()
counts = df_clean['diabetes'].value_counts().sort_index()
labels = ['No Diabetes (0)', 'Diabetes (1)']

plt.figure(figsize=(6, 4))
bars = plt.bar(labels, counts.values, color=['steelblue', 'tomato'], edgecolor='black')
for bar, val in zip(bars, counts.values):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 200, str(val), ha='center', fontsize=11)
plt.title('Class Distribution', fontsize=13)
plt.ylabel('Count')
plt.tight_layout()
plt.show()

print("Class counts:")
print(counts)
print(f"\nImbalance ratio: {counts[0]/counts[1]:.1f}:1 (Negative:Positive)")

**Observation:** The dataset is **highly imbalanced** — approximately **10.8:1 ratio** (82,284 non-diabetic vs 7,634 diabetic). This means models can achieve ~91% accuracy by simply predicting "no diabetes" always. We will use **stratified splitting** to ensure both classes are proportionally represented in train and test sets.

### 3.3 Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 9))

# Row 1: Numeric feature distributions by class
num_cols = ['age', 'bmi', 'HbA1c_level', 'blood_glucose_level']
for i, col in enumerate(num_cols):
    for label, grp in df_clean.groupby('diabetes')[col]:
        grp.dropna().plot(kind='hist', bins=30, alpha=0.6,
                          ax=axes[0, i], label=f'Class {int(label)}')
    axes[0, i].set_title(f'Distribution of {col}')
    axes[0, i].set_xlabel(col)
    axes[0, i].legend()

# Row 2: Categorical features vs Diabetes
cat_cols = ['gender', 'smoking_history', 'hypertension', 'heart_disease']
for i, col in enumerate(cat_cols):
    ct = df_clean.groupby([col, 'diabetes']).size().unstack(fill_value=0)
    ct.columns = ['No Diabetes', 'Diabetes']
    ct.plot(kind='bar', ax=axes[1, i], color=['steelblue', 'tomato'], edgecolor='black')
    axes[1, i].set_title(f'{col} vs Diabetes')
    axes[1, i].tick_params(axis='x', rotation=30)

plt.suptitle('Exploratory Data Analysis', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Pairplot of key numeric features (sample for speed)
sample = (df_clean[['age', 'bmi', 'HbA1c_level', 'blood_glucose_level', 'diabetes']]
          .dropna()
          .sample(1000, random_state=42)
          .copy())
sample['diabetes'] = sample['diabetes'].astype(int).astype(str)

sns.pairplot(sample, hue='diabetes',
             plot_kws={'alpha': 0.4, 's': 15},
             palette={'0': 'steelblue', '1': 'tomato'})
plt.suptitle('Pairplot of Key Features (1000 samples)', y=1.01)
plt.show()

## Section 4: Data Preprocessing

### 4.1 Fault: Missing Values
**Problem:** All 9 columns have approximately 10% missing values (~10,000 rows each).  
**Solution:** Drop all rows containing any null value. With 100,000 rows, dropping ~61,000 rows with any null still leaves ~38,636 clean rows — sufficient for training robust models. Imputation was considered but avoided because missing values appear across multiple columns simultaneously, making row-level imputation unreliable.

In [ ]:
print("Missing values BEFORE preprocessing:")
print(df.isnull().sum())
print(f"\nTotal rows: {len(df)}")

df_proc = df.dropna().copy()

print(f"\nShape after dropping nulls: {df_proc.shape}")
print("\nMissing values AFTER preprocessing:")
print(df_proc.isnull().sum())

### 4.2 Fault: Categorical Variables
**Problem:** `gender` (Female/Male/Other) and `smoking_history` (never/current/former/etc.) are text-based and cannot be used directly in ML models.  
**Solution:** Apply **Label Encoding**. Since neither column has a meaningful ordinal relationship needed for the model (tree-based models handle label-encoded categoricals well, and logistic regression / neural network work on the encoded integer representation), Label Encoding is appropriate here. One-Hot Encoding would also work but would increase dimensionality.

In [ ]:
print("Unique values before encoding:")
print("  gender:", df_proc['gender'].unique())
print("  smoking_history:", df_proc['smoking_history'].unique())

le = LabelEncoder()
df_proc['gender'] = le.fit_transform(df_proc['gender'])
df_proc['smoking_history'] = le.fit_transform(df_proc['smoking_history'])

print("\nAfter Label Encoding:")
print(df_proc[['gender', 'smoking_history']].head(8))

### 4.3 Fault: Feature Scaling
**Problem:** Features have very different scales — e.g., `blood_glucose_level` ranges 80–300, while `hypertension` is binary (0/1). Models like KNN, Logistic Regression, and Neural Networks are sensitive to feature scale.  
**Solution:** Apply **StandardScaler** (zero mean, unit variance). **Important:** The scaler is fit ONLY on training data and then applied to test data to prevent data leakage.

## Section 5: Dataset Splitting
- **Split type:** Stratified (to preserve class ratio due to imbalance)
- **Train set:** 80%
- **Test set:** 20%

In [ ]:
X = df_proc.drop('diabetes', axis=1)
y = df_proc['diabetes'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set size:  {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Test set size:      {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"\nTraining class distribution:")
print(y_train.value_counts())
print(f"\nTest class distribution:")
print(y_test.value_counts())

In [ ]:
# Apply StandardScaler — fit on TRAIN only
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print("Scaling applied.")
print(f"Feature means (train, after scaling): {X_train_sc.mean(axis=0).round(4)}")

## Section 6: Model Training & Testing

### 6.1 KNN Classifier

In [ ]:
knn = KNeighborsClassifier(n_neighbors=7)
knn.fit(X_train_sc, y_train)
y_pred_knn = knn.predict(X_test_sc)
y_prob_knn = knn.predict_proba(X_test_sc)[:, 1]

print("KNN Classification Report:")
print(classification_report(y_test, y_pred_knn, target_names=['No Diabetes', 'Diabetes']))

### 6.2 Decision Tree Classifier

In [ ]:
dt = DecisionTreeClassifier(max_depth=8, random_state=42)
dt.fit(X_train_sc, y_train)
y_pred_dt = dt.predict(X_test_sc)
y_prob_dt = dt.predict_proba(X_test_sc)[:, 1]

print("Decision Tree Classification Report:")
print(classification_report(y_test, y_pred_dt, target_names=['No Diabetes', 'Diabetes']))

### 6.3 Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_sc, y_train)
y_pred_lr = lr.predict(X_test_sc)
y_prob_lr = lr.predict_proba(X_test_sc)[:, 1]

print("Logistic Regression Classification Report:")
print(classification_report(y_test, y_pred_lr, target_names=['No Diabetes', 'Diabetes']))

### 6.4 Neural Network (MLPClassifier)
Architecture: **Input → 128 → 64 → 32 → Output (sigmoid)**  
Library: `sklearn.neural_network.MLPClassifier`  
Activation: ReLU, Early stopping enabled to prevent overfitting.

In [ ]:
nn = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),
    activation='relu',
    max_iter=300,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=15,
    verbose=False
)
nn.fit(X_train_sc, y_train)
y_pred_nn = nn.predict(X_test_sc)
y_prob_nn = nn.predict_proba(X_test_sc)[:, 1]

print(f"Training stopped after {nn.n_iter_} iterations.")
print("\nNeural Network Classification Report:")
print(classification_report(y_test, y_pred_nn, target_names=['No Diabetes', 'Diabetes']))

### 6.5 Unsupervised Learning: KMeans Clustering
The dataset is treated as an unsupervised problem. We use KMeans to discover natural groupings, then visualize them using PCA for dimensionality reduction.

In [ ]:
# Elbow method to find optimal k
inertias = []
K_range = range(2, 11)
for k in K_range:
    km_tmp = KMeans(n_clusters=k, random_state=42, n_init=10)
    km_tmp.fit(X_train_sc)
    inertias.append(km_tmp.inertia_)

plt.figure(figsize=(7, 4))
plt.plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia (Within-cluster Sum of Squares)')
plt.title('Elbow Method for Optimal k')
plt.tight_layout()
plt.show()

In [ ]:
# Fit KMeans with k=2 (matches binary class structure)
km = KMeans(n_clusters=2, random_state=42, n_init=10)
km.fit(X_train_sc)
km_labels = km.predict(X_test_sc)

# Visualize using PCA (2D projection)
pca = PCA(n_components=2, random_state=42)
X_test_pca = pca.fit_transform(X_test_sc)
print(f"PCA explained variance ratio: {pca.explained_variance_ratio_.round(3)}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(X_test_pca[:, 0], X_test_pca[:, 1],
                c=km_labels, cmap='coolwarm', alpha=0.4, s=5)
axes[0].set_title('KMeans Clusters (k=2)')
axes[0].set_xlabel('Principal Component 1')
axes[0].set_ylabel('Principal Component 2')

axes[1].scatter(X_test_pca[:, 0], X_test_pca[:, 1],
                c=y_test.values, cmap='coolwarm', alpha=0.4, s=5)
axes[1].set_title('True Labels (0=No Diabetes, 1=Diabetes)')
axes[1].set_xlabel('Principal Component 1')
axes[1].set_ylabel('Principal Component 2')

plt.suptitle('KMeans Clusters vs True Labels (PCA Projection)', fontsize=13)
plt.tight_layout()
plt.show()

## Section 7: Model Comparison & Evaluation

In [ ]:
# Collect all results
model_preds = {
    'KNN':                (y_pred_knn, y_prob_knn),
    'Decision Tree':      (y_pred_dt,  y_prob_dt),
    'Logistic Regression':(y_pred_lr,  y_prob_lr),
    'Neural Network':     (y_pred_nn,  y_prob_nn),
}

results = {}
for name, (y_pred, y_prob) in model_preds.items():
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    results[name] = {
        'Accuracy':  accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall':    recall_score(y_test, y_pred),
        'AUC':       auc(fpr, tpr)
    }

results_df = pd.DataFrame(results).T.round(4)
print("Model Comparison Summary:")
print(results_df)

### 7.1 Accuracy Comparison (Bar Chart)

In [ ]:
plt.figure(figsize=(8, 5))
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']
bars = plt.bar(results_df.index, results_df['Accuracy'],
               color=colors, edgecolor='black', width=0.5)
for bar, val in zip(bars, results_df['Accuracy']):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.001,
             f'{val:.4f}', ha='center', fontsize=10, fontweight='bold')
plt.ylim(0.90, 1.00)
plt.ylabel('Accuracy')
plt.title('Model Accuracy Comparison')
plt.tight_layout()
plt.show()

### 7.2 Precision & Recall Comparison

In [ ]:
x = np.arange(len(results_df))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, results_df['Precision'], width,
               label='Precision', color='steelblue', edgecolor='black')
bars2 = ax.bar(x + width/2, results_df['Recall'],    width,
               label='Recall',    color='tomato',    edgecolor='black')

for bar in bars1:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f'{bar.get_height():.3f}', ha='center', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f'{bar.get_height():.3f}', ha='center', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(results_df.index)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Precision & Recall Comparison')
ax.legend()
plt.tight_layout()
plt.show()

### 7.3 Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (name, (y_pred, _)) in zip(axes, model_preds.items()):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['No Diabetes', 'Diabetes'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontsize=11)
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.tick_params(axis='x', rotation=15)

plt.suptitle('Confusion Matrices for All Models', fontsize=14, y=1.03)
plt.tight_layout()
plt.show()

### 7.4 ROC Curves & AUC Scores

In [ ]:
plt.figure(figsize=(8, 6))
colors_roc = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

for (name, (y_pred, y_prob)), color in zip(model_preds.items(), colors_roc):
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=color, linewidth=2,
             label=f'{name} (AUC = {roc_auc:.4f})')

plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier (AUC = 0.5)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — All Models')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## Section 8: Conclusion

### Results Summary
| Model | Accuracy | Precision | Recall | AUC |
|---|---|---|---|---|
| KNN | ~95.8% | ~88.7% | ~57.3% | ~0.911 |
| Decision Tree | ~97.0% | ~96.5% | ~67.3% | ~0.971 |
| Logistic Regression | ~95.9% | ~86.7% | ~60.9% | ~0.956 |
| Neural Network | ~96.9% | ~96.0% | ~66.8% | ~0.973 |

### Interpretation
- **Neural Network** and **Decision Tree** achieved the best AUC scores (~0.97), indicating strong ability to distinguish diabetic from non-diabetic patients.
- **Precision is high across all models** for the diabetic class — when a model predicts diabetes, it is usually correct.
- **Recall is moderate (~57–67%)** — due to class imbalance (~8.5% positive cases), models sometimes miss diabetic patients. In a real clinical setting, maximizing recall (sensitivity) would be critical to avoid false negatives.
- **KMeans clustering** found two natural groups in the data that roughly correspond to the diabetic vs. non-diabetic split, validating the underlying structure of the dataset.

### Why These Results?
- `HbA1c_level` and `blood_glucose_level` dominate the decision boundary — these are the primary clinical indicators of diabetes.
- The high class imbalance (10:1 ratio) suppresses recall for the minority (diabetic) class.
- The Neural Network benefits from its ability to learn non-linear decision boundaries across all 8 features simultaneously.

### Challenges Faced
1. **High missing value rate** (~10% per column) — required dropping ~61% of rows, significantly reducing dataset size.
2. **Class imbalance** — made it necessary to use stratified splitting and pay close attention to precision/recall rather than accuracy alone.
3. **KMeans interpretability** — cluster labels are arbitrary; PCA visualization was needed to compare clusters to true labels.
4. **Neural Network training time** — larger hidden layers slow down training on 100K rows; early stopping was essential.